In [ ]:
# -*- coding: utf-8 -*-
import requests
import pandas as pd
# import package_installer as pi
# pi.import_or_install('plotly', '6.3.0')
# # Try to install kaleido without version constraint as it can be picky
# pi.import_or_install('kaleido', '1.2.0')
import plotly.graph_objects as go
from pathlib import Path


def get_available_counties(element: str) -> list:
    """Fetch available counties from Frost API."""
    r = requests.get(
        "https://frost.met.no/sources/v0.jsonld",
        params={
            "types": "SensorSystem",
            "elements": element
        },
        auth=("3acd117d-0ebb-4623-ac3a-e4d144a8868f", ""),
        timeout=5
    )
    r.raise_for_status()
    data = r.json().get("data", [])
    return sorted({
        s["county"]
        for s in data
        if s.get("county")
    })


def get_weather_stations(county: str, element: str) -> list:
    """Fetch weather stations for a given county."""
    r = requests.get(
        "https://frost.met.no/sources/v0.jsonld",
        params={
            "types": "SensorSystem",
            "county": county.lower(),
            "elements": element
        },
        auth=("3acd117d-0ebb-4623-ac3a-e4d144a8868f", ""),
        timeout=5
    )
    r.raise_for_status()
    data = r.json().get("data", [])
    return [
        f"{s['id']} – {s['name']}"
        for s in data
        if "id" in s and "name" in s
    ]

def read_pvt_file(path):
    """Read and parse PVT file."""
    with open(path, encoding="cp1252") as f:
        for i, line in enumerate(f):
            if line.strip().startswith("Datum") or line.strip().startswith("Date"):
                header_row = i
                break
    df = pd.read_csv(path, sep="\t", skiprows=header_row, encoding="cp1252")
    df = df.dropna(axis=1, how='all')
    # Normalize column names to handle both Swedish and English headers
    column_mapping = {
        'Date': 'Datum',
        'Time': 'Klockslag',
        'Absolute pressure': 'Absoluttryck',
        'Temperature': 'Temperatur',
        'Battery': 'Batteri'
    }
    df.rename(columns=column_mapping, inplace=True)
    df = df[df['Datum'].notna() & (df['Datum'].str.strip() != '')]
    df['datetime'] = pd.to_datetime(df['Datum'] + ' ' + df['Klockslag'], format='%Y-%m-%d %H:%M', errors='coerce')
    # Handle both comma and period as decimal separator
    df['Absoluttryck'] = df['Absoluttryck'].astype(str).str.replace(',', '.', regex=False)
    df['Absoluttryck'] = pd.to_numeric(df['Absoluttryck'], errors='coerce')
    df = df.sort_values('datetime')
    df['date'] = df['datetime'].dt.date
    return df

def read_excel_file(path, serial):
    """Read and parse Excel file with multiple sheets, selecting by serial number.
    
    Args:
        path: Path to Excel file
        serial: Serial number to identify which sheet to read
    
    Returns:
        DataFrame with processed data
    """
    # Read Excel file with the sheet named after the serial number
    df = pd.read_excel(path, sheet_name=str(serial), header=None)
    
    # Find the header row (look for row with "Serial" AND other data columns)
    header_row = None
    for i in range(min(30, len(df))):  # Check first 30 rows
        row_values = df.iloc[i].astype(str).str.strip().str.lower()
        row_values_original = df.iloc[i].astype(str).str.strip()
        
        # Check if this row contains typical column headers
        # Look for combinations that indicate this is the header row
        contains_serial = any('serial' in val for val in row_values)
        contains_datetime = any(val in ['date time', 'date', 'datum'] for val in row_values)
        contains_pressure = any(val in ['mmh2o', 'kpa', 'absoluttryck', 'absolute pressure'] for val in row_values)
        
        if contains_serial and (contains_datetime or contains_pressure):
            header_row = i
            break
    
    # Re-read with proper header
    if header_row is not None:
        df = pd.read_excel(path, sheet_name=str(serial), header=header_row)
    else:
        raise ValueError(f"Could not find header row with data columns in sheet {serial}. First 10 rows:\n{df.head(10).to_string()}")
    
    df = df.dropna(axis=1, how='all')
    
    # Drop the Serial column if it exists (we already know which sheet we're reading)
    if 'Serial' in df.columns:
        df = df.drop(columns=['Serial'])
    
    # Track which pressure unit we have for proper conversion later
    # Check before renaming to avoid duplicate columns
    pressure_unit = None
    pressure_col = None
    if 'mmH2O' in df.columns:
        pressure_unit = 'mmH2O'
        pressure_col = 'mmH2O'
    elif 'mH2O' in df.columns:
        pressure_unit = 'mH2O'
        pressure_col = 'mH2O'
    elif 'kPa' in df.columns:
        pressure_unit = 'kPa'
        pressure_col = 'kPa'
    
    # Normalize column names to handle different formats
    # Only map the pressure column that exists
    column_mapping = {
        'Date': 'Datum',
        'Time': 'Klockslag',
        'Date Time': 'datetime_str',  # Combined date time column
        'Absolute pressure': 'Absoluttryck',
        'Temperature': 'Temperatur',
        '°C': 'Temperatur',
        'Battery': 'Batteri',
        'Volt': 'Batteri'
    }
    
    # Add the pressure column to mapping if found
    if pressure_col:
        column_mapping[pressure_col] = 'Absoluttryck'
    
    df.rename(columns=column_mapping, inplace=True)
    
    # Handle datetime - check if we have combined datetime or separate date/time
    if 'datetime_str' in df.columns:
        # Combined Date Time column
        df['datetime'] = pd.to_datetime(df['datetime_str'], errors='coerce')
    elif 'Datum' in df.columns and 'Klockslag' in df.columns:
        # Separate Date and Time columns
        df['datetime'] = pd.to_datetime(df['Datum'].astype(str) + ' ' + df['Klockslag'].astype(str), 
                                         format='%Y-%m-%d %H:%M', errors='coerce')
    else:
        raise ValueError(f"Could not find datetime columns. Available columns: {df.columns.tolist()}")
    
    # Filter out empty rows
    df = df[df['datetime'].notna()]
    
    # Handle pressure - convert to mH2O (same unit as PVT files)
    if 'Absoluttryck' in df.columns:
        # Handle both comma and period as decimal separator
        df['Absoluttryck'] = df['Absoluttryck'].astype(str).str.replace(',', '.', regex=False)
        df['Absoluttryck'] = pd.to_numeric(df['Absoluttryck'], errors='coerce')
        
        # Convert based on original unit to mH2O
        if pressure_unit == 'mmH2O':
            # millimeters H2O to meters H2O: divide by 1000
            df['Absoluttryck'] = df['Absoluttryck'] / 1000
        elif pressure_unit == 'mH2O':
            # already in meters H2O, no conversion needed
            pass
        elif pressure_unit == 'kPa':
            # kPa to mH2O: 1 kPa ≈ 0.102 mH2O (or divide by 9.81)
            df['Absoluttryck'] = df['Absoluttryck'] / 9.81
        # If no unit detected, assume it's already in mH2O
    
    df = df.sort_values('datetime')
    df['date'] = df['datetime'].dt.date
    
    return df

def fetch_weather_data(station_id: str, start: str, end: str) -> pd.DataFrame:
    """Fetch weather data from Frost API."""
    endpoint = "https://frost.met.no/observations/v0.jsonld"
    params = {
        "sources": station_id,
        "elements": "sum(precipitation_amount P1D),mean(air_pressure_at_sea_level P1D)",
        "referencetime": f"{start}/{end}"
    }
    r = requests.get(
        endpoint, 
        params=params, 
        auth=("3acd117d-0ebb-4623-ac3a-e4d144a8868f", "")
    )
    r.raise_for_status()
    data = r.json()

    rain = []
    for item in data.get("data", []):
        date = item["referenceTime"][:10]
        precip = None
        lufttrykk = None
        for obs in item["observations"]:
            if obs["elementId"] == "sum(precipitation_amount P1D)":
                precip = obs.get("value")
            elif obs["elementId"] == "mean(air_pressure_at_sea_level P1D)":
                lufttrykk = obs.get("value")
        rain.append({"date": date, "rain_mm": precip, "lufttrykk": lufttrykk})

    rain_df = pd.DataFrame(rain)
    rain_df["date"] = pd.to_datetime(rain_df["date"]).dt.date
    return rain_df


def calculate_pressure(df: pd.DataFrame, rain_df: pd.DataFrame, measurement_level: float) -> pd.DataFrame:
    """Calculate corrected pressure and trykkhøyde."""
    df = pd.merge(df, rain_df, on="date", how="left")
    df["absoluttrykk_kPa"] = df["Absoluttryck"] / 10194 * 100000
    df["Absoluttryck_korr"] = df["absoluttrykk_kPa"] - df["lufttrykk"] * 100 / 1000
    df["Trykkhøyde"] = df["Absoluttryck_korr"] * 1000 / 9.81 / 1000 + measurement_level
    return df


def create_plot(df: pd.DataFrame, rain_df: pd.DataFrame, terrain_level: float, borepoint: str, measurement_level: float, installation_depth: float, station_id: str, serial_number: str) -> go.Figure:
    """Create plotly figure for poretrykk data."""
    fig = go.Figure()

    max_trykkhoyde = df['Trykkhøyde'].max()
    min_trykkhoyde = df['Trykkhøyde'].min()
    max_idx = df['Trykkhøyde'].idxmax()
    min_idx = df['Trykkhøyde'].idxmin()
    mean_trykkhoyde = df['Trykkhøyde'].mean()
    mean_idx = (df['Trykkhøyde'] - mean_trykkhoyde).abs().idxmin()

    fig.add_trace(go.Scatter(
        x=df['datetime'],
        y=df['Trykkhøyde'],
        name="Trykkhøyde (m)",
        mode="lines",
        line=dict(dash='solid', color='crimson'),
        yaxis="y1"
    ))

    fig.add_trace(go.Bar(
        x=pd.to_datetime(rain_df["date"]),
        y=rain_df["rain_mm"],
        name="Nedbør (mm)",
        yaxis="y2",
        opacity=0.5,
        marker_color='blue'
    ))

    fig.add_trace(go.Scatter(
        x=(df['datetime'].iloc[0], df['datetime'].iloc[-1]),
        y=(terrain_level, terrain_level),
        name="Terrengnivå [m]",
        mode="lines",
        line=dict(dash='dash', color='black')
    ))

    fig.add_trace(go.Scatter(
        x=[df.loc[max_idx, 'datetime']],
        y=[max_trykkhoyde],
        mode='markers',
        name=f'Maks: {max_trykkhoyde:.1f} m',
        marker=dict(color='red', size=12, symbol='triangle-up')
    ))

    fig.add_trace(go.Scatter(
        x=[df.loc[min_idx, 'datetime']],
        y=[min_trykkhoyde],
        mode='markers',
        name=f'Min: {min_trykkhoyde:.1f} m',
        marker=dict(color='green', size=12, symbol='triangle-down')
    ))

    fig.add_trace(go.Scatter(
        x=[df.loc[mean_idx, 'datetime']],
        y=[mean_trykkhoyde],
        mode='markers',
        name=f'Gj.snitt: {mean_trykkhoyde:.1f} m',
        marker=dict(color='rgba(255, 255, 255, 0.8)', size=0, symbol='circle')
    ))
    # Create info box text
    info_text = (
        f"<b>Måleinformasjon:</b><br>"
        f"Terrengnivå: {terrain_level:.1f} m<br>"
        f"Installasjonsdybde: {installation_depth:.1f} m<br>"
        f"Målenivå: {measurement_level:.1f} m<br>"
        f"Værstasjon: {station_id}<br>"
        f"Lufttrykkskorrigert: Ja"
    )

    fig.add_annotation(
        xref="paper", yref="paper",
        x=0,  # Left edge (0 to 1 scale)
        y=-0.075, # Below the legend
        text=info_text,
        showarrow=False,
        align="left",
        bgcolor="rgba(255, 255, 255, 0.9)",
        # bordercolor="black",
        # borderwidth=1,
        borderpad=10,
        font=dict(size=11),
        xanchor="left",
        yanchor="top"
    )

    fig.update_layout(
        title=dict(
            xref="paper", yref="paper",
            text=f"{borepoint} - Poretrykksregistering - Målernr {serial_number}",
            x=0,
            xanchor='left'
        ),
        xaxis=dict(
            title="Dato (-)", 
            tickformat="%d.%m.%Y",
            showline=True,
            linewidth=2,
            linecolor='grey',
            mirror=True
        ),
        yaxis=dict(
            title="Kotenivå (m)", 
            side="left",
            showline=True,
            linewidth=2,
            linecolor='grey',
            mirror=True,
            showgrid=True,
            gridcolor='lightgrey',
            griddash='dash'
        ),
        yaxis2=dict(
            title="Nedbør (mm)",
            overlaying="y",
            side="right",
            showgrid=False,
            title_standoff=25,
            ticks="inside",
            ticklen=5,
            ticklabelstandoff=15,
            automargin=True
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0,
            bgcolor="rgba(255, 255, 255, 0.8)",
            
        ),
        plot_bgcolor='white',
        paper_bgcolor='white',
        margin=dict(b=175, r=120, l=120, t=150),
        width=1123,   # A4 landscape width in pixels
        height=794    # A4 landscape height in pixels
    )
    return fig

def process_poretrykk(
    pvt_path: str,
    serial_number: str,
    station_id: str,
    terrain_level: float,
    installation_depth: float,
    borepoint: str
) -> str:
    """Main processing function."""
    measurement_level = terrain_level - installation_depth

    # Determine file type and read data accordingly
    if pvt_path.lower().endswith('.xlsx') or pvt_path.lower().endswith('.xls'):
        # Excel file - requires serial number
        if not serial_number:
            raise ValueError("Serial number is required for Excel files")
        df = read_excel_file(pvt_path, serial_number)
    else:
        # PVT file
        df = read_pvt_file(pvt_path)

    # Fetch weather data
    start = df['datetime'].min().strftime('%Y-%m-%d')
    end = df['datetime'].max().strftime('%Y-%m-%d')
    rain_df = fetch_weather_data(station_id, start, end)

    # Calculate pressure
    df = calculate_pressure(df, rain_df, measurement_level)

    # Create plot
    fig = create_plot(df, rain_df, terrain_level, borepoint, measurement_level, installation_depth, station_id, serial_number)

    # Save figure
    output_dir = Path(pvt_path).parent
    output_file_html = output_dir / f"{borepoint}_poretrykk_plot.html"
    output_file_pdf = output_dir / f"{borepoint}_poretrykk_plot.pdf"
    
    fig.write_html(str(output_file_html))
    fig.show()
    # Try to save as PDF
    try:
        import arcpy
        has_arcpy = True
    except ImportError:
        has_arcpy = False
    
    try:
        # Attempt PDF generation (kaleido is used automatically)
        # scale parameter helps avoid "Kaleido-fier" header artifact
        fig.write_image(str(output_file_pdf), width=1123, height=794, scale=1)
        
        # Check if PDF was actually created
        if output_file_pdf.exists():
            if has_arcpy:
                arcpy.AddMessage(f"PDF saved to: {output_file_pdf}")
        else:
            if has_arcpy:
                arcpy.AddWarning("PDF generation reported success but file not found")
    except Exception as e:
        # PDF generation failed
        if has_arcpy:
            arcpy.AddWarning(f"PDF generation failed: {str(e)}")
            arcpy.AddMessage("HTML file is available as alternative")

    return str(output_file_html)

# Example usage
process_poretrykk(
    pvt_path= r"C:\Users\jdr\OneDrive - Multiconsult\Skrivebord\Test_PZ_plot\fv109 - Copy.xlsx",
    serial_number="39434",
    station_id="SN17000",
    terrain_level=25.0,
    installation_depth=6.0,
    borepoint="4-15")

'C:\\Users\\jdr\\OneDrive - Multiconsult\\Skrivebord\\Test_PZ_plot\\4-15_poretrykk_plot.html'